# Child Activity Monitor on Google Colab

Run the cells in order with **Shift+Enter**. This notebook:
1. sets up the project and the AI models,
2. analyses your own video clips (reading, gadget use, playing, eating, idle/sleeping) and shows the labelled videos and accuracy,
3. opens the full dashboard (report + "Test a video" page) in a new tab.

Colab cannot use your webcam or CCTV camera live; that part runs on the laptop or Raspberry Pi.

## 1. Upload the project zip
Choose `child-activity-monitor.zip` when the file picker appears.

In [ ]:
from google.colab import files
import os, zipfile
up = files.upload()
name = next(n for n in up if n.endswith('.zip'))
zipfile.ZipFile(name).extractall('/content')
%cd /content/child-activity-monitor
!ls

## 2. Install libraries and download the models (about 2 minutes)

In [ ]:
!apt-get -qq install -y libegl1 libgles2 > /dev/null
!pip -q install mediapipe flask firebase-admin "opencv-python>=4.8,<5" pytest
!python scripts/download_models.py

## 3. Check everything works
All tests should pass.

In [ ]:
!python -m pytest -q

## 4. Upload your test videos
Name each file after the activity it shows, e.g. `reading_1.mp4`, `gadget_use_phone.mp4`, `playing_ball.mp4`, `eating_lunch.mp4`, `idle_sleeping_nap.mp4`, `running_1.mp4`, `jumping_1.mp4`, `lying_down_1.mp4`, `falling_1.mp4`, `zoned_out_1.mp4`, `crying_1.mp4`.
(Group and fight clips need two or more children: test those on the dashboard's Test a video page, which lets you pick the children from left to right.)
The name is only used to score accuracy; any other name is analysed but not scored.

In [ ]:
import os, shutil
from google.colab import files
os.makedirs('test_videos', exist_ok=True)
for n in files.upload():
    shutil.move(n, os.path.join('test_videos', n))
!ls -la test_videos

## 5. Analyse the videos
Each clip is processed at 5 frames per second, like on the Raspberry Pi.
Set `CHILD_NAME` to your name first: it is written on the box in each labelled video.

In [ ]:
import os, base64
import pandas as pd
from IPython.display import HTML, display
from monitor import ACTIVITIES
from monitor.config import load_config
from monitor.video_analysis import analyse_video

# The name written on the box in the labelled videos: change it to your own name.
CHILD_NAME = 'Your name'

cfg = load_config()
cfg['privacy']['blur_faces'] = False   # True = blur faces (privacy mode)
os.makedirs('labelled', exist_ok=True)
rows = []
for name in sorted(os.listdir('test_videos')):
    expected = next((a for a in sorted(ACTIVITIES, key=len, reverse=True) if name.lower().startswith(a)), None)
    out = os.path.join('labelled', os.path.splitext(name)[0] + '.webm')
    try:
        r = analyse_video(os.path.join('test_videos', name), cfg, annotate_path=out,
                         children=['me'], names={'me': CHILD_NAME})
    except ValueError as e:
        print(name, 'skipped:', e); continue
    total = sum(r['counts'].values())
    rows.append({'clip': name, 'expected': expected or '', 'predicted': r['predicted'],
                 'correct': (r['predicted'] == expected) if expected else None,
                 **{k: round(100 * v / total) for k, v in r['counts'].items()},
                 'objects seen': ', '.join(r['objects'])})
    print(f"{name}: {r['predicted']}" + (f"  ({'OK' if r['predicted'] == expected else 'WRONG'}, expected {expected})" if expected else ''))

df = pd.DataFrame(rows).fillna(0)
display(df)
scored = df[df['expected'] != '']
if len(scored):
    print(f"\nAccuracy: {int(scored['correct'].sum())}/{len(scored)} = {100 * scored['correct'].mean():.0f}%")
    display(pd.crosstab(scored['expected'], scored['predicted'], rownames=['expected'], colnames=['predicted']))

## 6. Watch the labelled videos
Boxes = detected person/objects, yellow dots = pose, top-left = detected activity.

In [ ]:
for name in sorted(os.listdir('labelled')):
    data = base64.b64encode(open(os.path.join('labelled', name), 'rb').read()).decode()
    display(HTML(f'<h4>{name}</h4><video width=480 controls src="data:video/webm;base64,{data}"></video>'))

## 7. Open the dashboard (optional)
Loads a week of demo data, then opens the dashboard in a new tab. Sign in as **admin / admin123** (or parent1 / parent123, teacher / teacher123). Stop it with the ■ button.

In [ ]:
!python scripts/seed_demo_data.py --reset
import subprocess, time
server = subprocess.Popen(['python', '-m', 'dashboard.app'])
time.sleep(4)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(5000)

## 8. Download the labelled videos and results (optional)

In [ ]:
df.to_csv('results.csv', index=False)
!zip -qr labelled_results.zip labelled results.csv
files.download('labelled_results.zip')